In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib as plt 
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

ModuleNotFoundError: No module named 'tensorflow'

In [326]:
df= pd.read_csv("IMDb Movies India.csv",encoding= "latin1")

In [327]:
df.head(5)

,Name,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
0,,NaN,NaN,Drama,NaN,NaN,J.S. Randhawa,Manmauji,Birbal,Rajendra Bhatia
1,#Gadhvi (He thought he was Gandhi),(2019),109 min,Drama,7.0,8,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid
2,#Homecoming,(2021),90 min,"Drama, Musical",NaN,NaN,Soumyajit Majumdar,Sayani Gupta,Plabita Borthakur,Roy Angana
3,#Yaaram,(2019),110 min,"Comedy, Romance",4.4,35,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor
4,...And Once Again,(2010),105 min,Drama,NaN,NaN,Amol Palekar,Rajat Kapoor,Rituparna Sengupta,Antara Mali


In [328]:
df.isnull().sum()

Name           0
Year         528
Duration    8269
Genre       1877
Rating      7590
Votes       7589
Director     525
Actor 1     1617
Actor 2     2384
Actor 3     3144
dtype: int64

In [329]:
df.shape

(15509, 10)

In [330]:
# df["Rating"] = df["Rating"].dropna(inplace = True)
df = df.dropna(subset=['Rating'])

In [331]:
df.isnull().sum()

Name           0
Year           0
Duration    2068
Genre        102
Rating         0
Votes          0
Director       5
Actor 1      125
Actor 2      200
Actor 3      292
dtype: int64

In [332]:
df.isnull()

,Name,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
1,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False
5,False,False,False,False,False,False,False,False,False,False
6,False,False,False,False,False,False,False,False,False,False
8,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...
15501,False,False,True,False,False,False,False,False,False,False
15503,False,False,False,False,False,False,False,False,False,False
15504,False,False,True,False,False,False,False,False,False,False
15505,False,False,False,False,False,False,False,False,False,False


In [333]:
df.head(5)

,Name,Year,Duration,Genre,Rating,Votes,Director,Actor 1,Actor 2,Actor 3
1,#Gadhvi (He thought he was Gandhi),(2019),109 min,Drama,7.0,8,Gaurav Bakshi,Rasika Dugal,Vivek Ghamande,Arvind Jangid
3,#Yaaram,(2019),110 min,"Comedy, Romance",4.4,35,Ovais Khan,Prateik,Ishita Raj,Siddhant Kapoor
5,...Aur Pyaar Ho Gaya,(1997),147 min,"Comedy, Drama, Musical",4.7,827,Rahul Rawail,Bobby Deol,Aishwarya Rai Bachchan,Shammi Kapoor
6,...Yahaan,(2005),142 min,"Drama, Romance, War",7.4,"1,086",Shoojit Sircar,Jimmy Sheirgill,Minissha Lamba,Yashpal Sharma
8,?: A Question Mark,(2012),82 min,"Horror, Mystery, Thriller",5.6,326,Allyson Patel,Yash Dave,Muntazir Ahmad,Kiran Bhatia


In [334]:
df.isnull().sum()


Name           0
Year           0
Duration    2068
Genre        102
Rating         0
Votes          0
Director       5
Actor 1      125
Actor 2      200
Actor 3      292
dtype: int64

In [335]:
df["Duration"] = df['Duration'].str.replace(" min","",regex = False).astype(float)
df["Duration"].fillna(df["Duration"].median(), inplace= True)
df['Votes'] = pd.to_numeric(df['Votes'].str.replace(',', ''))

In [336]:
df['Genre'].fillna(df['Genre'].mode()[0],inplace = True)

In [337]:
Y = df["Rating"]

In [338]:
df.isnull().sum()

Name          0
Year          0
Duration      0
Genre         0
Rating        0
Votes         0
Director      5
Actor 1     125
Actor 2     200
Actor 3     292
dtype: int64

In [339]:
df["Actor 1"] = df["Actor 1"].fillna(df["Actor 1"].mode()[0])
df["Actor 2"] = df["Actor 2"].fillna(df["Actor 2"].mode()[0])
df["Actor 3"] = df["Actor 3"].fillna(df["Actor 3"].mode()[0])

In [340]:
df.drop("Year", axis=1,inplace = True)

In [341]:
genre_dummies = df['Genre'].str.get_dummies(sep=', ')

df = pd.concat([df.drop('Genre', axis=1), genre_dummies], axis=1)

In [343]:
# One-Hot Encoding
categorical_cols = ['Director', 'Actor 1', 'Actor 2', 'Actor 3']

df = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

In [344]:
df.dtypes


Name                       object
Duration                  float64
Rating                    float64
Votes                       int64
Action                      int64
                           ...   
Actor 3_Zeishan Quadri       bool
Actor 3_Zenobia Shroff       bool
Actor 3_Zohra                bool
Actor 3_Zoya Hussain         bool
Actor 3_Zulfi Sayed          bool
Length: 11649, dtype: object

In [345]:
df.head(5)

,Name,Duration,Rating,Votes,Action,Adventure,Animation,Biography,Comedy,Crime,...,Actor 3_Zarine Ali,Actor 3_Zayed Khan,Actor 3_Zebunissa,Actor 3_Zeenat Aman,Actor 3_Zeeshan Khan,Actor 3_Zeishan Quadri,Actor 3_Zenobia Shroff,Actor 3_Zohra,Actor 3_Zoya Hussain,Actor 3_Zulfi Sayed
1,#Gadhvi (He thought he was Gandhi),109.0,7.0,8,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
3,#Yaaram,110.0,4.4,35,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,False,False
5,...Aur Pyaar Ho Gaya,147.0,4.7,827,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,False,False
6,...Yahaan,142.0,7.4,1086,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False
8,?: A Question Mark,82.0,5.6,326,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,False,False


In [346]:
X = df.drop(columns=["Rating",'Name'])

In [347]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7919 entries, 1 to 15508
Columns: 11647 entries, Duration to Actor 3_Zulfi Sayed
dtypes: bool(11623), float64(1), int64(23)
memory usage: 89.3 MB


In [348]:
X_train , X_test , y_train , y_test  = train_test_split(X, Y , test_size= 0.2, random_state= 42)

In [ ]:
# model = RandomForestRegressor(
#     n_estimators=500,
#     max_depth=20,
#     min_samples_split=5,
#     min_samples_leaf=2,
#     random_state=42,
#     n_jobs=-1
# )

# model.fit(X_train , y_train)



# Build Neural Network Model
model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    Dense(32, activation="relu"),
    Dense(1, activation="linear")  # Linear activation for regression
])

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.001), loss="mean_squared_error", metrics=["mae"])

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",20
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

In [350]:
y_pred = model.predict(X_test)

In [351]:
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))


MAE: 0.9418223020126537
MSE: 1.4310818089177744


In [352]:
r2 = r2_score(y_test, y_pred)
print("R2 Score:", r2)

R2 Score: 0.2302473983159915


In [353]:
df.isnull().sum()

Name                      0
Duration                  0
Rating                    0
Votes                     0
Action                    0
                         ..
Actor 3_Zeishan Quadri    0
Actor 3_Zenobia Shroff    0
Actor 3_Zohra             0
Actor 3_Zoya Hussain      0
Actor 3_Zulfi Sayed       0
Length: 11649, dtype: int64